In [13]:
%pip install mlxtend

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [14]:
import pandas as pd 
from mlxtend.frequent_patterns import apriori, association_rules
df = pd.read_csv('Assignment-1_Data.csv', encoding='ISO-8859-1', sep=';')
print(df.head())

  ï»¿BillNo                             Itemname  Quantity              Date  \
0    536365   WHITE HANGING HEART T-LIGHT HOLDER         6  01.12.2010 08:26   
1    536365                  WHITE METAL LANTERN         6  01.12.2010 08:26   
2    536365       CREAM CUPID HEARTS COAT HANGER         8  01.12.2010 08:26   
3    536365  KNITTED UNION FLAG HOT WATER BOTTLE         6  01.12.2010 08:26   
4    536365       RED WOOLLY HOTTIE WHITE HEART.         6  01.12.2010 08:26   

  Price  CustomerID         Country  
0  2,55     17850.0  United Kingdom  
1  3,39     17850.0  United Kingdom  
2  2,75     17850.0  United Kingdom  
3  3,39     17850.0  United Kingdom  
4  3,39     17850.0  United Kingdom  


C:\Users\dell\AppData\Local\Temp\ipykernel_34508\4129855092.py:3: DtypeWarning: Columns (0: ï»¿BillNo) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('Assignment-1_Data.csv', encoding='ISO-8859-1', sep=';')


### **Step 1: Data Loading and Inspection**
* **Objective:** Import the retail transaction dataset and inspect its structure, column names, and data types to prepare for Market Basket Analysis.
* **Method:** Utilizing the Pandas library with semicolon separation (`sep=';'`) and appropriate encoding to handle special characters.

In [15]:
print('--- Data Info ---')
print(df.info())

--- Data Info ---
<class 'pandas.DataFrame'>
RangeIndex: 522064 entries, 0 to 522063
Data columns (total 7 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   ï»¿BillNo   522064 non-null  object 
 1   Itemname    520609 non-null  str    
 2   Quantity    522064 non-null  int64  
 3   Date        522064 non-null  str    
 4   Price       522064 non-null  str    
 5   CustomerID  388023 non-null  float64
 6   Country     522064 non-null  str    
dtypes: float64(1), int64(1), object(1), str(4)
memory usage: 27.9+ MB
None


In [16]:
df = df.rename(columns={df.columns[0]: 'BillNo'})
df = df.dropna(subset=['Itemname', 'CustomerID'])
df = df[~df['BillNo'].astype(str).str.startswith('C')]
print("--- Data Cleaned Successfully ---")
print("Remaining rows and columns:", df.shape)

--- Data Cleaned Successfully ---
Remaining rows and columns: (388023, 7)


### **Step 2: Data Cleaning and Preprocessing**
* **Objective:** Clean the dataset by handling missing descriptions, filtering out cancelled transactions (invoices starting with 'C'), and renaming columns for consistency.
* **Method:** Dropping null values in essential columns and using string filtering techniques.

In [17]:
basket = (df[df['Country'] == 'France']
          .groupby(['BillNo', 'Itemname'])['Quantity']
          .sum().unstack().reset_index().fillna(0)
          .set_index('BillNo'))

def encode_units(x):
    return 1 if x >= 1 else 0

basket_sets = basket.map(encode_units)

print("--- Basket Formatted Successfully ---")
print(basket_sets.head(3))

--- Basket Formatted Successfully ---
Itemname  10 COLOUR SPACEBOY PEN  12 COLOURED PARTY BALLOONS  \
BillNo                                                         
536370                         0                           0   
536852                         0                           0   
536974                         0                           0   

Itemname  12 EGG HOUSE PAINTED WOOD  12 MESSAGE CARDS WITH ENVELOPES  \
BillNo                                                                 
536370                            0                                0   
536852                            0                                0   
536974                            0                                0   

Itemname  12 PENCIL SMALL TUBE WOODLAND  12 PENCILS SMALL TUBE RED RETROSPOT  \
BillNo                                                                         
536370                                0                                    0   
536852                                0 

### **Observation & Explanation:**
* **Basket Matrix Creation:** Grouped the transactions by `BillNo` and `Itemname` for a specific region (France) to structure the data into a market basket format.
* **Binary Encoding:** Converted product quantities into binary values (`1` for purchased, `0` for not purchased), which is the standard requirment for running the Apriori algorithm.

In [18]:
from mlxtend.frequent_patterns import apriori, association_rules
frequent_itemsets = apriori(basket_sets, min_support=0.05, use_colnames=True)

rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1)

print("--- Top Association Rules ---")
print(rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(10))

--- Top Association Rules ---
                                antecedents  \
0  frozenset({4 TRADITIONAL SPINNING TOPS})   
1                      frozenset({POSTAGE})   
2   frozenset({ALARM CLOCK BAKELIKE GREEN})   
3    frozenset({ALARM CLOCK BAKELIKE PINK})   
4   frozenset({ALARM CLOCK BAKELIKE GREEN})   
5     frozenset({ALARM CLOCK BAKELIKE RED})   
6                      frozenset({POSTAGE})   
7   frozenset({ALARM CLOCK BAKELIKE GREEN})   
8    frozenset({ALARM CLOCK BAKELIKE PINK})   
9     frozenset({ALARM CLOCK BAKELIKE RED})   

                                consequents   support  confidence      lift  
0                      frozenset({POSTAGE})  0.056555    0.785714  1.018810  
1  frozenset({4 TRADITIONAL SPINNING TOPS})  0.056555    0.073333  1.018810  
2    frozenset({ALARM CLOCK BAKELIKE PINK})  0.074550    0.763158  7.421711  
3   frozenset({ALARM CLOCK BAKELIKE GREEN})  0.074550    0.725000  7.421711  
4     frozenset({ALARM CLOCK BAKELIKE RED})  0.079692    0.815

c:\Users\dell\AppData\Local\Programs\Python\Python313\Lib\site-packages\mlxtend\frequent_patterns\fpcommon.py:175: DeprecationWarning: DataFrames with non-bool types result in worse computationalperformance and their support might be discontinued in the future.Please use a DataFrame with bool type
  warnings.warn(


### **Step 4: Association Rules & Business Insights**
* **Apriori Execution:** Successfully generated frequent itemsets and association rules using a minimum support threshold of 5% (`min_support=0.05`) and lift metrics.
* **Key Observations:** 
  * Rules with high **Lift** values (e.g., matching colorful Alarm Clocks like Pink, Green, and Red) indicate strong product affinities and purchasing patterns among customers.
  * These insights can be directly used by retail platforms for cross-selling, product recommendations, and creating attractive bundle offers.

In [19]:
print('Null values before cleaning :\n', df.isnull().sum())
df_clean = df.dropna(subset=['Itemname', 'CustomerID']).copy()

Null values before cleaning :
 BillNo        0
Itemname      0
Quantity      0
Date          0
Price         0
CustomerID    0
Country       0
dtype: int64


### **1. Handling Missing Values**
* **Objective:** Remove rows with missing essential information such as product descriptions (`Itemname`) or customer identifiers (`CustomerID`).
* **Result:** Ensures data integrity for grouping and association rule mining.

In [20]:
df_clean = df_clean[~df_clean['BillNo'].astype(str).str.startswith('C')]
print('Shape after removing cancellation:', df_clean.shape)

Shape after removing cancellation: (388023, 7)


### **2. Filtering Cancelled Transactions**
* **Objective:** Exclude transactions where orders were cancelled (indicated by 'C' in the bill number).
* **Result:** Keeps only successful purchase records for accurate basket analysis.

In [21]:
df_clean = df_clean[df_clean['Quantity'] > 0]
print('fianl cleaned dataset shape:', df_clean.shape)

fianl cleaned dataset shape: (388023, 7)


### **3. Filtering Valid Quantities**
* **Objective:** Ensure all transaction quantities are positive integers greater than zero.
* **Result:** Prepares a clean, error-free dataset ready for one-hot encoding and matrix transformation.

In [22]:
null_counts = df.isnull().sum()
duplicate_count = df.duplicated().sum()
dtypes = df.dtypes
quality_report = pd.DataFrame({
    'Null_Count': null_counts,
    'Data_Type':  dtypes
})
print('--- DATA QUALITY REPORT ---')
print(quality_report)
print(f"\nTotal Duplicate Rows: {duplicate_count}")
print(f'Total Dataset Shape (Rows, Columns): {df.shape}')

--- DATA QUALITY REPORT ---
            Null_Count Data_Type
BillNo               0    object
Itemname             0       str
Quantity             0     int64
Date                 0       str
Price                0       str
CustomerID           0   float64
Country              0       str

Total Duplicate Rows: 5210
Total Dataset Shape (Rows, Columns): (388023, 7)


### **1. Data Quality Report**
* **Objective:** Access the intial state of the raw dataset by identifyingmissing values, duplicate entries, and datatype inconsistencies.
* **Result:** Provided a complete breakdown aof null counts per columns and total duplicates to guide the cleaning strategy.

In [24]:
initial_rows = df.shape[0]
df_cleaned_task3 = df.drop_duplicates().copy()
duplicates_removed = initial_rows - df_cleaned_task3.shape[0]
print(f'Initial Row Count: {initial_rows}')
print(f'Duplicates Removed: {duplicates_removed}')
print(f'Row Count After Removing Duplicates: {df_cleaned_task3.shape[0]}')

Initial Row Count: 388023
Duplicates Removed: 5210
Row Count After Removing Duplicates: 382813


### **2 Duplicate Removal**
* **Objective:** Identify and eliminate redundant duplicate records to prevent bias in analysis.
* **Result:** Successfully removed duplicate rows and tracked the exact count of dropped records.

In [25]:
Q1 = df_cleaned_task3['Quantity'].quantile(0.25)
Q3 = df_cleaned_task3['Quantity'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
outliers_count = df_cleaned_task3[(df_cleaned_task3['Quantity'] < lower_bound) | (df_cleaned_task3['Quantity'] > upper_bound).shape[0]]
print(f'Detected Outliers in Quantity using IQR: {outliers_count}')

Detected Outliers in Quantity using IQR:         BillNo                             Itemname  Quantity  \
0       536365   WHITE HANGING HEART T-LIGHT HOLDER         6   
1       536365                  WHITE METAL LANTERN         6   
2       536365       CREAM CUPID HEARTS COAT HANGER         8   
3       536365  KNITTED UNION FLAG HOT WATER BOTTLE         6   
4       536365       RED WOOLLY HOTTIE WHITE HEART.         6   
...        ...                                  ...       ...   
522059  581587          PACK OF 20 SPACEBOY NAPKINS        12   
522060  581587          CHILDREN'S APRON DOLLY GIRL         6   
522061  581587         CHILDRENS CUTLERY DOLLY GIRL         4   
522062  581587      CHILDRENS CUTLERY CIRCUS PARADE         4   
522063  581587         BAKING SET 9 PIECE RETROSPOT         3   

                    Date Price  CustomerID         Country  
0       01.12.2010 08:26  2,55     17850.0  United Kingdom  
1       01.12.2010 08:26  3,39     17850.0  United Kingd

### **3. Outliers Detection**
* **Objective** Use the Interquantile Range (IQR) method to flag extrreme anomalous values in numeric columns.
* **Result** Quantified outliers to dicide whether to retain or filter them based on business logic.

In [26]:
summary_data = {
    'Metric': ['Total Rows', 'Duplicate Rows', 'Missing Values (Itemname)'],
    'Before Cleaning': [df.shape[0], df.duplicated().sum(), df['Itemname'].isnull().sum()],
    'After Cleaning': [df_cleaned_task3.shape[0], df_cleaned_task3.duplicated().sum(), df_cleaned_task3['Itemname'].isnull().sum()]
}
summary_df = pd.DataFrame(summary_data)
print('--- BEFORE VS AFTER SUMMARY TABLE ---')
print(summary_df)
df_cleaned_task3.to_csv('cleaned_retail_dataset.csv', index=False)
print("\nCleaned dataset successfully saved as 'cleaned_retail_dataset.csv'!")

--- BEFORE VS AFTER SUMMARY TABLE ---
                      Metric  Before Cleaning  After Cleaning
0                 Total Rows           388023          382813
1             Duplicate Rows             5210               0
2  Missing Values (Itemname)                0               0

Cleaned dataset successfully saved as 'cleaned_retail_dataset.csv'!


### **4.  Summary & Export**
* **Before vs. After Comparison:** Documented matric improvement including row counts, null eliminations, and duplicate clearances.
* **File Export:** Stored the final analysis-ready dataset into a CSV file (`cleaned_retail_dataset.csv`).